# Custom CNN (KneeCNN) – comparison of data methods

In this notebook we train our own CNN, **KneeCNN**, from scratch and compare three ways of handling the class imbalance. The code follows `02_transfer_learning.ipynb` (same sections, same data code, same training loop, same plots), so both notebooks can be compared directly.

Data methods compared (same model, same settings, same seed):
- `none` – no augmentation (the first data cell of `02_transfer_learning.ipynb`)
- `rotation_balanced` – Jobin's rotation augmentation (the commented-out code in `02_transfer_learning.ipynb`)
- `mild_aug_sampler` – mild augmentation + weighted sampler (the data cell `02_transfer_learning.ipynb` uses)

Quick test: `USE_SAMPLE_DATA = True`. Real comparison: `USE_SAMPLE_DATA = False` (Colab with a T4 GPU).

## Import everything in colab

The following cell sets up the project in a fresh Colab runtime. This clones the repository, loads the dependencies and configures settings.

In [ ]:
!git clone https://github.com/michael-duer/medical-image-classification.git
%cd medical-image-classification

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Set paths
# NEW: True = 150 sample X-rays (quick test), False = real dataset
USE_SAMPLE_DATA = True

if USE_SAMPLE_DATA:
    DATA_DIR = Path('/content/medical-image-classification/example_data/knee_oa_test')
    if not DATA_DIR.exists():
        import zipfile
        zipfile.ZipFile('/content/medical-image-classification/example_data/knee_oa_test.zip').extractall('/content/medical-image-classification/example_data')
    # The sample zip was made on a Mac: delete the hidden "._" files, they are not images
    for mac_file in DATA_DIR.rglob('._*'):
        mac_file.unlink()
else:
    DATA_DIR = Path('/content/medical-image-classification/data')
OUTPUT_DIR = Path('/content/medical-image-classification/output')

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Set DATA_DIR to your actual dataset folder first: {DATA_DIR}")
print('Dataset path:', DATA_DIR.resolve())

## 1. Load and inspect data

The dataset already split the data into `train`, `test` and `val` folders. The same label mapping and normalization will be used by every model. The number of images between classes varies a lot. We compare three ways to handle this; only the training data differs, validation and test are never augmented.

`make_loaders(method)` contains the data code as `02_transfer_learning.ipynb`, one block per method.his way we can train KneeCNN with every method under the same conditions and choose the one that fits best

In [ ]:
from torchvision.datasets.folder import default_loader
from src.data_augmentation import BalancedAugmentedDataset

# Normalization matches common ImageNet-pretrained ResNet/EfficientNet weights.
# Keep these choices the same in all experiments.
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 5

class ImageSubset(Dataset):
    """ImageFolder-compatible subset that stores explicit (path, label) entries."""
    def __init__(self, samples, transform):
        self.samples = [(str(p), int(y)) for p, y in samples]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = default_loader(path)  # RGB
        return self.transform(img), label

# Load datasets
train_source = datasets.ImageFolder(DATA_DIR / 'train')
val_source = datasets.ImageFolder(DATA_DIR / 'val')
test_source = datasets.ImageFolder(DATA_DIR / 'test')

split_samples = {
        'train': train_source.samples,
        'val': val_source.samples,
        'test': test_source.samples,
    }

class_names = train_source.classes
num_classes = len(class_names)

# No random augmentation for validation/test
base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Mild augmentation for knee X-rays
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


def make_loaders(method):
    if method == 'none':
        # First data cell of 02_transfer_learning.ipynb: no augmentation
        datasets_by_split = {
            split: ImageSubset(samples, base_transform)
            for split, samples in split_samples.items()
        }
        loaders = {
            split: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                              pin_memory=(DEVICE.type == 'cuda'))
            for split, ds in datasets_by_split.items()
        }

    elif method == 'rotation_balanced':
        # Commented-out code of 02_transfer_learning.ipynb: apply augmentation to training data to handle class imbalance
        train_dataset = BalancedAugmentedDataset(
            split_samples["train"],
            image_size=IMAGE_SIZE,
            max_rotation=5,
            seed=SEED,
            mean=MEAN,
            std=STD
        )

        # Validation and test datasets: unchanged
        val_dataset = ImageSubset(
            split_samples["val"],
            base_transform
        )

        test_dataset = ImageSubset(
            split_samples["test"],
            base_transform
        )

        datasets_by_split = {
            "train": train_dataset,
            "val": val_dataset,
            "test": test_dataset
        }

        loaders = {
            split: DataLoader(
                ds,
                batch_size=BATCH_SIZE,
                shuffle=(split == "train"),
                pin_memory=(DEVICE.type == "cuda")
            )
            for split, ds in datasets_by_split.items()
        }

    elif method == 'mild_aug_sampler':
        # Second data cell of 02_transfer_learning.ipynb: mild augmentation + weighted sampler
        train_dataset = ImageSubset(
            split_samples["train"],
            train_transform
        )

        val_dataset = ImageSubset(
            split_samples["val"],
            base_transform
        )

        test_dataset = ImageSubset(
            split_samples["test"],
            base_transform
        )

        # Build weighted sampler from original training labels
        labels = np.array([
            label for _, label in split_samples["train"]
        ])

        counts = np.bincount(labels, minlength=NUM_CLASSES)

        # Softer balancing than 1/count
        alpha = 1
        class_weights = 1.0 / (counts ** alpha)
        sample_weights = class_weights[labels]

        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(
                sample_weights,
                dtype=torch.double
            ),
            num_samples=len(labels),
            replacement=True
        )

        loaders = {
            "train": DataLoader(
                train_dataset,
                batch_size=BATCH_SIZE,
                sampler=sampler,
                pin_memory=(DEVICE.type == "cuda")
            ),
            "val": DataLoader(
                val_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                pin_memory=(DEVICE.type == "cuda")
            ),
            "test": DataLoader(
                test_dataset,
                batch_size=BATCH_SIZE,
                shuffle=False,
                pin_memory=(DEVICE.type == "cuda")
            )
        }

    else:
        raise ValueError(f'Unknown method: {method}')

    return loaders


# Methods to compare (remove one to skip it)
METHODS = ['none', 'rotation_balanced', 'mild_aug_sampler']

print("Original class counts:", np.bincount([label for _, label in split_samples["train"]], minlength=NUM_CLASSES))
for method in METHODS:
    print(f"{method}: training batches per epoch:", len(make_loaders(method)["train"]))

## 2. Custom CNN trained from scratch

Unlike ResNet-18 and EfficientNet-B0, KneeCNN starts with random weights: all layers are trained, no freezing.

### 2.1 Set Training parameters
Specify learning rate, loss function and number of training loops (epochs). Also specify the number of classes to predict. The same settings are used for every data method.

In [ ]:
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10
NUM_CLASSES = 5
DROPOUT = 0.5
criterion = nn.CrossEntropyLoss()

Function to run training loop:

In [ ]:
def run_epoch(model, loader, optimizer=None, epoch=None, num_epochs=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss, correct, count = 0.0, 0, 0

    phase = "Train" if is_training else "Validate"
    description = f"Epoch [{epoch}/{num_epochs}] | {phase}"

    with tqdm(loader, leave=False, desc=description) as progressbar:
      for x, y in progressbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(x)
            loss = criterion(logits, y)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        count += y.size(0)

        progressbar.set_postfix(
            loss=f"{total_loss / count:.4f}",
            acc=f"{correct / count:.2%}"
        )

    return {'loss': total_loss / count, 'accuracy': correct / count}

### 2.2 KneeCNN
Our custom CNN instead of a pretrained model

In [ ]:
def conv_block(in_ch, out_ch):
    """Two 3x3 convolutions with BatchNorm + ReLU, then halve the image size."""
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class KneeCNN(nn.Module):
    """Custom CNN trained from scratch. Input: 3x224x224 image, output: one score per grade."""

    def __init__(self, n_classes, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # 224 -> 112
            conv_block(32, 64),    # 112 -> 56
            conv_block(64, 128),   # 56  -> 28
            conv_block(128, 256),  # 28  -> 14
            conv_block(256, 256),  # 14  -> 7
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),   # raw scores, no softmax
        )

    def forward(self, x):
        return self.classifier(self.features(x))


print('Learnable parameters:', sum(p.numel() for p in KneeCNN(NUM_CLASSES).parameters()))

## 2.2.1 Training + validation loops

The same training loop as for ResNet-18 in `02_transfer_learning.ipynb`, run once per data method. Before each method the seed is reset, so every method starts from the same random model.

In [ ]:
histories = {}
all_loaders = {}

from src.metrics import plot_training_history

for method in METHODS:
    print(f"\n========== {method} ==========")

    # Same start for every method
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    loaders = make_loaders(method)
    all_loaders[method] = loaders

    # Create KneeCNN with 5 output classes and move it to the gpu
    model = KneeCNN(NUM_CLASSES, dropout=DROPOUT).to(DEVICE)

    # Train all parameters (no pretrained layers to freeze)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    history = []
    best_val_loss = float("inf")
    # Stops model after 4 epochs of not improving validation loss
    early_stop_rate = 4
    epochs_without_improvement = 0
    # Reduce learning rates when validation loss stagnates
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    for epoch in range(1, NUM_EPOCHS + 1):
        train_stats = run_epoch(
            model, loaders['train'], optimizer,
            epoch=epoch, num_epochs=NUM_EPOCHS
        )

        val_stats = run_epoch(
            model, loaders['val'],
            epoch=epoch, num_epochs=NUM_EPOCHS
        )

        history.append({
          'epoch': epoch,
          'train_loss': train_stats['loss'],
          'train_accuracy': train_stats['accuracy'],
          'val_loss': val_stats['loss'],
          'val_accuracy': val_stats['accuracy']
        })

        tqdm.write(
          f"Epoch [{epoch}/{NUM_EPOCHS}] | "
          f"Train Loss: {train_stats['loss']:.4f} | "
          f"Train Acc: {train_stats['accuracy']:.2%} | "
          f"Val Loss: {val_stats['loss']:.4f} | "
          f"Val Acc: {val_stats['accuracy']:.2%}"
        )

        scheduler.step(val_stats['loss'])

        # Stop training if validation loss does not improve
        if val_stats['loss'] < best_val_loss:
            best_val_loss = val_stats['loss']
            epochs_without_improvement = 0
            # Save best model
            torch.save(model.state_dict(), f"best_kneecnn_{method}.pth")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stop_rate:
            print("Early stopping triggered!")
            break

    histories[method] = history

    # Plot training metrics over epochs
    plot_training_history(
        history,
        output_dir=OUTPUT_DIR / "metrics",
        model_name=f"KneeCNN ({method})",
        show_plots = True,
        save_plots = False
    )

## 2.2.2 Validation predictions and metrics

Each method's best model (lowest validation loss, as in `02_transfer_learning.ipynb`) is tested on the validation set. AUROC is calculated as in `02_transfer_learning.ipynb`: softmax probabilities, one-vs-rest per grade, macro average.

New here: we compare three data methods; the one with the highest validation AUROC is used for the final test (section 7)

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    actual, probs = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        actual.append(y.numpy())
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(actual), np.concatenate(probs)

def summarize_metrics(y_true, probabilities, class_names):
    predictions = probabilities.argmax(axis=1)
    results = {'accuracy': accuracy_score(y_true, predictions)}
    present = np.unique(y_true)
    if len(class_names) == 2 and len(present) == 2:
        results['auroc'] = roc_auc_score(y_true, probabilities[:, 1])
    elif len(present) == len(class_names):
        results['auroc'] = roc_auc_score(
            y_true, probabilities, labels=np.arange(len(class_names)),
            multi_class='ovr', average='macro')
    else:
        results['auroc'] = float('nan')
        print('AUROC undefined: not all classes appear in this validation split.')
    return results, confusion_matrix(y_true, predictions,
                                      labels=np.arange(len(class_names)))

comparison = []
for method in METHODS:
    model = KneeCNN(NUM_CLASSES, dropout=DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(f"best_kneecnn_{method}.pth", map_location=DEVICE, weights_only=True))

    y_val, val_prob = get_predictions(model, all_loaders[method]['val'])
    metrics, cm = summarize_metrics(y_val, val_prob, class_names)
    print(f'\n===== {method} =====')
    print('Validation metrics:', metrics)
    print('Confusion matrix (rows=true, cols=predicted):')
    display(pd.DataFrame(cm, index=class_names, columns=class_names))

    comparison.append({
        'method': method,
        'val_auroc': metrics['auroc'],
        'val_accuracy': metrics['accuracy'],
        'best_val_loss': min(h['val_loss'] for h in histories[method]),
        'epochs_run': len(histories[method]),
    })

comparison = pd.DataFrame(comparison).sort_values('val_auroc', ascending=False).reset_index(drop=True)
display(comparison)

BEST_METHOD = comparison.loc[0, 'method']
print('Best method (highest validation AUROC):', BEST_METHOD)

best_model = KneeCNN(NUM_CLASSES, dropout=DROPOUT).to(DEVICE)
best_model.load_state_dict(torch.load(f"best_kneecnn_{BEST_METHOD}.pth", map_location=DEVICE, weights_only=True))

## **7**. Detailed CNN evaluation

This uses the **test set**: run it only once, when the comparison is finished. It evaluates the best method from 2.2.2.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

plt.switch_backend("module://matplotlib_inline.backend_inline")

print("Current backend:", matplotlib.get_backend())

# Generate predictions
y_test, test_prob = get_predictions(best_model, all_loaders[BEST_METHOD]["test"])

print("True labels:", y_test.shape)
print("Predicted probabilities:", test_prob.shape)

from src.metrics import evaluate_predictions

# Evaluate CNN performance
results = evaluate_predictions(
    y_true=y_test,
    y_prob=test_prob,
    model_name=f"KneeCNN ({BEST_METHOD})",
    show_plots=True,
    save_plots=False
)

print(results)